In [ ]:
import glob
import os

import pandas as pd
import numpy as np

import tifffile as tf
from tifffile import imread
from tifffile import imwrite

import skimage
from skimage.util import compare_images
from skimage import io, color, measure
from skimage.measure import regionprops
from skimage.measure import label as sk_label  # Renaming the label function to avoid conflicts
from skimage.morphology import disk, dilation
from scipy.ndimage import distance_transform_edt

import matplotlib.pyplot as plt
import matplotlib

from tqdm import tqdm

%load_ext autoreload
%autoreload 2
%matplotlib inline

In [272]:
# Import images

images_dir = '/Volumes/Franklin/20241012/ACTB-KI_NGN2_Batch20241005_D7/Puro10min/Zstack'
images_path = os.path.join(images_dir,'*.stk') 
images_files = glob.glob(images_path)

In [ ]:
print(images_dir)
print(images_path)
print(len(images_files))

In [ ]:
# Sorts images in list alphabetically

images_files_sorted = np.sort(images_files)

print(len(images_files_sorted))
images_files_sorted

In [275]:
images_files_halo = [f for f in images_files_sorted if 'conf561' in f]
images_files_tubulin = [f for f in images_files_sorted if 'conf488' in f]
images_files_actin = [f for f in images_files_sorted if 'conf640' in f]

In [ ]:
# Initialize lists to store images by channel
images_tubulin = []
images_actin = []

# Sort images into respective lists based on filename cues
for file in tqdm(images_files_sorted):
    if 'conf488' in file:
        images_tubulin.append(imread(file))
    elif 'conf640' in file:
        images_actin.append(imread(file))

# Check the lists
print(f"Tubulin images: {len(images_tubulin)}")
print(f"Actin images: {len(images_actin)}")

In [ ]:
np.unique(images_actin[0])

In [ ]:
# Function to extract the center slice
def extract_center_slice(image):
    """
    Extract the center slice of a 3D image stack.

    Parameters:
    image (numpy.ndarray): A 3D numpy array representing the image stack 
                           (shape: slices x height x width).

    Returns:
    numpy.ndarray: The center slice (2D array).
    """
    center_index = image.shape[0] // 2  # Calculate the center slice index
    return image[center_index]


# Apply max projection to each list
center_slice_tubulin = [extract_center_slice(img) for img in images_tubulin]
center_slice_actin = [extract_center_slice(img) for img in images_actin]

# Check the number of max projections created
print(f"Single slices for Tubulin: {len(center_slice_tubulin)}")
print(f"Single slices for Actin: {len(center_slice_actin)}")

In [ ]:
# Plot whole cell and nucleus mask side by side

fig, ax = plt.subplots(1, 2, figsize = (8,4))
ax[0].imshow(center_slice_tubulin[2])
ax[1].imshow(center_slice_actin[2])

In [ ]:
# Plot whole cell and nucleus mask side by side

fig, ax = plt.subplots(1, 3, figsize = (12,4))
ax[0].imshow(center_slice_actin[0])
ax[1].imshow(center_slice_actin[1])
ax[2].imshow(center_slice_actin[2])

In [ ]:
# Filtering of high intensity values and filling holes with pixel values around holes

# Set threshold to identify high-intensity pixels
threshold_value = 6000  # Adjust this based on your image's intensity range

# Structuring element for dilation
selem = disk(5)

center_slice_actin_processed = []

for idx, img in enumerate(center_slice_actin):
    # Mask for high-intensity pixels
    high_intensity_mask = img > threshold_value

    # Dilate mask to include neighboring pixels
    expanded_mask = dilation(high_intensity_mask, selem)

    # Set high-intensity pixels to 0 in filtered image
    img_filtered = img.copy()
    img_filtered[expanded_mask] = 0

    # Fill 0 pixels inside the mask using neighboring pixel values from outside mask
    img_filled = img_filtered.copy()

    zero_mask = img_filled == 0  # Pixels that need filling

    # Distance transform to calculate the distance to nearest non-zero pixels
    dist_transform = distance_transform_edt(zero_mask)

    # Fill the mask from outside in using the average of the valid neighbors
    # Propagate the nearest valid pixels into the 0 pixels
    for r in range(1, img_filled.shape[0] - 1):
        for c in range(1, img_filled.shape[1] - 1):
            if zero_mask[r, c]:
                # Calculate neighbors from outside the mask
                neighbors = []
                for nr in range(r - 1, r + 2):
                    for nc in range(c - 1, c + 2):
                        if 0 <= nr < img_filled.shape[0] and 0 <= nc < img_filled.shape[1]:
                            if img_filled[nr, nc] != 0:
                                neighbors.append(img_filled[nr, nc])
                if neighbors:
                    img_filled[r, c] = np.mean(neighbors)

    center_slice_actin_processed.append(img_filled)

    # Display the original, filtered, and filled images
    plt.figure(figsize=(15, 7))
    plt.subplot(131), plt.imshow(img, cmap='gray'), plt.title(f'Original Image {idx+1}')
    plt.subplot(132), plt.imshow(img_filtered, cmap='gray'), plt.title(f'Filtered Image {idx+1}')
    plt.subplot(133), plt.imshow(img_filled, cmap='gray'), plt.title(f'Filled Image {idx+1}')
    plt.show()

In [ ]:
len(center_slice_actin_processed)

In [286]:
# Output directory max projections

single_slice_dir = r'/Volumes/bamfaile/Analysis/ACTB-KI/00_Batch20241005_iNs/D7/Puro10min/Center_slice'


In [ ]:
# Save max projections to seperate folders

# Define the folder names
folders = ['Tubulin', 'Actin']

# Create the folders if they don't exist
for folder in folders:
    folder_path = os.path.join(single_slice_dir, folder)
    os.makedirs(folder_path, exist_ok=True)

# Function to save a max projection image
def save_max_projection_image(image, original_file, channel_name):
    # Extract the file name (without path)
    file_name = os.path.basename(original_file)
    
    # Extract the base name (remove everything after the last '_')
    base_name = file_name.rsplit('_', 1)[0]  # This splits at the last underscore
    
    # Create the new filename with the appropriate suffix (e.g., "_tubulin", "_actin")
    new_file_name = base_name + f"_{channel_name}.tif"
    
    # Define the output path
    output_file = os.path.join(single_slice_dir, channel_name, new_file_name)
    
    # Check if file already exists to prevent overwriting
    if os.path.exists(output_file):
        print(f"Warning: {output_file} already exists! Skipping...")
        return
    
    # Save the max projection image to the corresponding folder
    tf.imwrite(output_file, image)
    print(f"Saved {output_file}")

# Now save the max projections for each channel

for i, image_file in enumerate(images_files_tubulin):
    save_max_projection_image(center_slice_tubulin[i], image_file, 'tubulin')

for i, image_file in enumerate(images_files_actin):
    save_max_projection_image(center_slice_actin_processed[i], image_file, 'actin')
